In [1]:
from hdawg_init import *
import zhinst

import numpy as np
import matplotlib.pyplot as plt
import time
import zhinst.core
from datetime import datetime
import pyvisa as visa
from datetime import datetime
import json

from RFSoC_PARAMPconfig_08012024_dysl_v1 import *

In [133]:
ip = "TCPIP0::192.168.0.27::inst0::INSTR"
rm = visa.ResourceManager()
kna = rm.open_resource(ip)



In [3]:
daq = zhinst.core.ziDAQServer('127.0.0.1', 8004, 6)
# Starting module awgModule on 2024/02/26 14:07:29
# awg_zh_init(daq)
# init_code_paramp(daq)

In [94]:
#%% Configure the board

# # Create board object 
board = RFSoC_auxill("192.168.0.210",userconfig=None,debug=True,logging=False)

Connection Established.
Version 2020.2
RfdcVersion rfdc_v8.1
RFSoC PARAMP CLASS


228 is DAC tile 0

229 is DAC tile 1

230 is DAC tile 2

231 is DAC tile 3


Each Tile has 4 DACs - 0,1,2,3

Name convention on board

0_228 => DAC 0 of Tile 0

2_229 => DAC 2 of Tile 1

1st index on all list of lists indicate tile

2nd index on all list of lists indicate DAC



In [95]:
# DAC Tile and HDAWG DAC defs

#[[Tile,DAC],HDAWG_DAC]

ch_map = { "rr1": [[0,0],1],
          "rr2": [[1,2],2],
          "rr3": [[1,0],3],
          "rr4": [[0,2],4],
          "rr5": [[0,1],5],
          "rr6": [[0,3],6],
          "rr7": [[1,3],7],
          "rr8": [[1,1],8]
}

# just run the following w/o thinking

In [96]:
#%% USER AND BOARD PARAMETERS

# DUAL TONE PROPERTY DECLARATION
# IF PLAN
# f_c = [[0e6,0e6,0e6,0e6],[0e6,0e6,0e6,0e6]] # Centre Frequency declaration. 
f_sep = [[1600e6,1600e6,1600e6,1600e6],[1600e6,1600e6,1600e6,1600e6]] # Frequency Separation of dual tones

# DAC TILE and BLOCK PLAN
Tiles = [[0, 0, 0, 0],[1, 1, 1, 1]]
Blocks = [[0, 1, 2, 3],[0, 1, 2, 3]]
Type = 1

# BOARD DAC FIFO SAMPLING RATES
F_sample = 8847.36#8355.84 # MHz # RF sampling Frequency

F_sample1 = [[8847.36,8847.36,8847.36,8847.36]]

Interpolation = 2 # 4: 2x inerpolation
fs_IF = F_sample*1e6/Interpolation # Sampling Frequency of IF
# BOARD DAC PROPERTIES (FOR DIRECT SETTING)
# f_firstNyq = 1355.84 # MHz

Cavity_freqs = [[0,0,0,0],[0,0,0,0]]


# PYTHON CODE PARAMETERS
wait_in_sec = 0.0
# NUMBER OF SAMPLES IN I OR Q CHANNEL OF BRAM
N = 8192 * 1
# PREALLOCATION
wave_data = [[0,0,0,0],[0,0,0,0]]        

In [97]:
#Cavity Frequency Setting

# in MHz

cavity_frequencies = {
    "rr1": 7.404596e3,
    "rr2": 7.385942e3,
    "rr3": 7.426435e3,
    "rr4": 7.314808e3,
    "rr5": 7.356673e3,
    "rr6": 7.355762e3,
    "rr7": 7.12446e3,
    "rr8": 7.41618e3,
}

In [98]:
cavity_freq_extract(cavity_frequencies=cavity_frequencies)

[[7404.596, 7385.942, 7426.435, 7314.808],
 [7356.673, 7355.762, 7124.46, 7416.18]]

Cavity Frequency Setting

In [99]:
# AMPLITUDE PLAN
Amp1_offset = [[0,0,0,0],[0,0,0,0]] # DC Offset
Amp1 = [[0.5,0.5,0.43,0.43],[0.38,0.45,0.4,0.55]]        # Sideband imbalance
tot_pwr = [[0.0,0.0,0.275,0.275],[0,0.0,0.0,0.0]]

# DAC CURRENT PLAN
DAC_currents = [[14.75,14.5,21.25,31.75],[28.75,20,14,20]]       # currents in mA

DC_bias = [[-1.1,0,0,0],[0,0,0,0]]



# cavity_freqs = [[7.41618e3,7.3e3,4.5e3,7.3e3],[7.41618e3,7.3e3,7.3e3,7.3e3]]
# cavity_freqs = cavity_freq_extract(cavity_frequencies=cavity_frequencies)

# for i in range(2):
#     for j in range(4):
#         Cavity_freqs[i][j] = F_sample - cavity_freqs[i][j]

for rr, freq in cavity_frequencies.items():
    t_dac = ch_map[f'rr{rr[-1]}'][0][1]
    t_tile = ch_map[f'rr{rr[-1]}'][0][0]
    # print(f't_dac = {t_dac} t_tile = {t_tile}')
    Cavity_freqs[t_tile][t_dac] = F_sample - freq

# int for Internal source and ext for External Source

In [100]:
board_init(board=board,int_ext='ext',F_sample=F_sample) # Only touch the int ext for clock source

TCS file read, beginning clock programming.
SetStartEndClkConfig 1
rfclkWriteReg 0 2 0 144
rfclkWriteReg 0 2 0 16
rfclkWriteReg 0 2 2 0
rfclkWriteReg 0 2 3 6
rfclkWriteReg 0 2 4 208
rfclkWriteReg 0 2 5 91
rfclkWriteReg 0 2 6 0
rfclkWriteReg 0 2 12 81
rfclkWriteReg 0 2 13 4
rfclkWriteReg 0 2 256 116
rfclkWriteReg 0 2 257 85
rfclkWriteReg 0 2 258 85
rfclkWriteReg 0 2 259 1
rfclkWriteReg 0 2 260 34
rfclkWriteReg 0 2 261 0
rfclkWriteReg 0 2 262 115
rfclkWriteReg 0 2 263 3
rfclkWriteReg 0 2 264 106
rfclkWriteReg 0 2 265 85
rfclkWriteReg 0 2 266 85
rfclkWriteReg 0 2 267 0
rfclkWriteReg 0 2 268 34
rfclkWriteReg 0 2 269 0
rfclkWriteReg 0 2 270 240
rfclkWriteReg 0 2 271 48
rfclkWriteReg 0 2 272 116
rfclkWriteReg 0 2 273 85
rfclkWriteReg 0 2 274 85
rfclkWriteReg 0 2 275 1
rfclkWriteReg 0 2 276 34
rfclkWriteReg 0 2 277 0
rfclkWriteReg 0 2 278 115
rfclkWriteReg 0 2 279 3
rfclkWriteReg 0 2 280 106
rfclkWriteReg 0 2 281 85
rfclkWriteReg 0 2 282 85
rfclkWriteReg 0 2 283 1
rfclkWriteReg 0 2 284 34
rfc

# just run this w/o thinking

In [101]:
#%% GENERATE DATA (just run this w/o thinking)

for i in range(len(Blocks)):
    for j in range(len(Blocks[i])):
        [wave_data[i][j],_,_] = gen_pulses(fs_IF, N, 0, f_sep[i][j], Amp1_offset[i][j], Amp1[i][j],tot_pwr[i][j])

# Programming beings. Set rr number and pump = 1/0 for on/off

* # Initialisation. Start with tot_pwr = 0.5 and Amp1 = 0.5. Amp1 is the lower sideband power fraction i.e. if Amp1=1, then signal-wise same power in both sidebands. If DAC current adjustment reaches 32, and still not enough gain, increase tot_pwr and start lower again.

* # f_sep = 0 for single pump and f_sep = 1600e6 for double pump at 800e6 gap

* # Can optimise sideband power ratio to get same gain at lower pump power

* # Saturates at tot_pwr1 = 0.28 and DAC Current = 32 mA for all DACs. TAKE CARE 

In [415]:
#%% PROGRAM DAC one by one

rr = 6

#[[Tile,DAC],HDAWG_DAC]

DAC_no = ch_map[f'rr{rr}'][0][1]
Tile = ch_map[f'rr{rr}'][0][0]


hdawg_daq_ch = ch_map[f'rr{rr}'][1]

current = 10#30.75


In [417]:
#### Saturates at tot_pwr1 = 0.25 for all DACs. TAKE CARE
pump =  1  #pump on/off
print(f'Pump state = {pump}')

if pump:
    tot_pwr_adj = 0.275
    Amp1_adj = 0.46
    [data,_,_] = gen_pulses(fs_IF=fs_IF,N=N, f_c=0, f_sep=1600e6, Amp1_offset=0.0, Amp1=Amp1_adj,tot_pwr1=tot_pwr_adj)##DOUBLE PUMP
    # [data,_,_] = gen_pulses(fs_IF=fs_IF,N=N, f_c=0, f_sep=0, Amp1_offset=0.0, Amp1=0.48,tot_pwr1=0.275)##SINGLE PUMP
    # DAC_prog(board, 1, Tile, DAC_no, F_sample, 32, Cavity_freqs[DAC_no][Tile], data)
    DAC_prog(board, 1, Tile, DAC_no, F_sample, current, Cavity_freqs[Tile][DAC_no], data)
else:
    [data,_,_] = gen_pulses(fs_IF, N, 0, 1600e6, 0, 0.5,0.0)
    DAC_prog(board, 1, Tile, DAC_no, F_sample, 8, Cavity_freqs[Tile][DAC_no], data)
    

Pump state = 1
 Programming DAC 3_228 (DAC 3, Tile 0)...
SetMemType
SetDataPathMode 0 3 2 
SetMMCM 1 5 2 5 0 
SetNyquistZone
SetupFIFO
SetInterpolationFactor
SetMMCM 1 5 2 5 0 
IntrClr
SetupFIFO
MultiBand 1 0 1 4 1
SetMixerSettings
ResetNCOPhase
UpdateEvent
SetDACVOP
SetDecoderMode
SetInvSincFir
DAC 3_228 (DAC 3, Tile 0) settings programmed.
Sending data to DAC 3, Tile 0. (3_228)
LocalMemInfo 1 b0000000 4 32 16384 16 65535 0
LocalMemTrigger
SetLocalMemSample
------------------------------COMMAND SENT----------------------------
-----------------------------DATA SENT--------------------------------
WriteDataToMemory
LocalMemTrigger
LocalMemTrigger
LocalMemInfo 1 b0000000 4 32 16384 16 65535 0
Data Sent. 3_228 DAC active (DAC 3, Tile 0)
SetDACVOP
DATA UPLOADED


# HDAWG offset adjustment

# Current Adjustment

In [409]:
#%% POST CALIBRATION STATUS CHECK / UPDATE
current = 31.25
change_current(board,current,DAC_no,Tile)
print(f'Current current = {current}')

SetDACVOP
Current current = 31.25


In [410]:
offset = 0.13
# (daq, ch, V_fin, V_init=None, step=0.01, t_wait=1, verbose=False):
ramp_V_hdawg(daq,ch=hdawg_daq_ch, step = 0.1, V_fin = offset)

/dev8099/sigouts/5/offset
/dev8099/sines/5/amplitudes/1
V_init automatically taken from current setting of offset = 0.12 and amplitude = 0.0
values 0.12, 0.13, 0.1, 0.010000000000000009
/dev8099/sigouts/5/offset0.12
/dev8099/sines/5/amplitudes/10
/dev8099/sigouts/5/offset0.125
/dev8099/sines/5/amplitudes/10
/dev8099/sigouts/5/offset0.13
/dev8099/sines/5/amplitudes/10


In [411]:
# Save settings


DAC_currents[Tile][DAC_no] = current
tot_pwr[Tile][DAC_no] = tot_pwr_adj
Amp1[Tile][DAC_no] = Amp1_adj

DC_bias[Tile][DAC_no] = read_offset(daq,hdawg_daq_ch) + read_amp(daq,hdawg_daq_ch)





/dev8099/sigouts/5/offset
/dev8099/sines/5/amplitudes/1


In [379]:
str1 = '/dev8099/sigouts/'
str2 = '/on'
str = str1 + f'{5}' + str2
print(str)

daq.getInt(str)

/dev8099/sigouts/5/on


1

In [364]:
DAC_currents, DC_bias

([[14.75, 14.5, 31.5, 31.75], [22.75, 20, 14, 20]],
 [[-1.1, 0, -1.8, 0], [-1.0, 0, 0, 0]])

In [412]:
## Save with VNA data into file

gain_profile = (kna.query_ascii_values("CALC1:MEAS1:DATA:FDAT?"))
time.sleep(4)
f_data = (kna.query_ascii_values("CALC1:MEAS1:X:VAL?"))

gain_data = list(np.transpose([f_data, gain_profile]))

save_data = {}
save_data['DAC_currents'] = current
save_data['tot_pwr'] = tot_pwr_adj
save_data['Amp1'] = Amp1_adj
save_data['DC_bias'] = offset

now = datetime.now()
current_date = now.strftime("%y-%m-%d")

file_name = f'paramp_gain_data_{current_date}_R{(2*rr+5)%12}'

with open(f'./paramp_data_files/{file_name}.json','w') as f:
    json.dump(save_data, f, indent = 6)
    f.close()

np.savetxt(fname = f'./paramp_data_files/{file_name}.csv', X = gain_data)

In [269]:
save_data

{'DAC_currents': 22.75, 'tot_pwr': 0.275, 'Amp1': 0.47, 'DC_bias': -1}

In [ ]:
#Print final settings
DAC_currents, tot_pwr, Amp1, DC_bias


In [420]:
## All pymp off

for i in range(2):
    for j in range(4):
        [data,_,_] = gen_pulses(fs_IF, N, 0, 1600e6, 0, 0.5,0.0)
        DAC_prog(board, 1, Tile, DAC_no, F_sample, 8, Cavity_freqs[i][j], data)

 Programming DAC 3_228 (DAC 3, Tile 0)...


OSError: [WinError 10038] An operation was attempted on something that is not a socket

In [418]:
board.break_connection()

Connection Closed.


In [419]:
board

In [ ]:
# AMPLITUDE PLAN
Amp1_offset = [[0,0,0,0],[0,0,0,0]] # DC Offset
Amp1 = [[0.5,0.5,0.43,0.43],[0.38,0.45,0.4,0.55]]        # Sideband imbalance
tot_pwr = [[0.0,0.0,0.275,0.275],[0,0.0,0.0,0.0]]

# DAC CURRENT PLAN
DAC_currents = [[14.75,14.5,21.25,31.75],[28.75,20,14,20]]       # currents in mA

In [106]:
## All pymp on simultaneously

for i in range(2):
    for j in range(4):
        [data,_,_] = gen_pulses(fs_IF, N, 0, 1600e6, 0, Amp1[i][j],tot_pwr[i][j])
        DAC_prog(board, 1, Tile, DAC_no, F_sample, DAC_currents[i][j], Cavity_freqs[i][j], data)

 Programming DAC 1_228 (DAC 1, Tile 0)...
SetMemType
SetDataPathMode 0 1 2 
SetMMCM 1 5 2 5 0 
SetNyquistZone
SetupFIFO
SetInterpolationFactor
SetMMCM 1 5 2 5 0 
IntrClr
SetupFIFO
MultiBand 1 0 1 4 1
SetMixerSettings
ResetNCOPhase
UpdateEvent
SetDACVOP
SetDecoderMode
SetInvSincFir
DAC 1_228 (DAC 1, Tile 0) settings programmed.
Sending data to DAC 1, Tile 0. (1_228)
LocalMemInfo 1 b0000000 4 32 16384 16 65535 0
LocalMemTrigger
SetLocalMemSample
------------------------------COMMAND SENT----------------------------
-----------------------------DATA SENT--------------------------------
WriteDataToMemory
LocalMemTrigger
LocalMemTrigger
LocalMemInfo 1 b0000000 4 32 16384 16 65535 0
Data Sent. 1_228 DAC active (DAC 1, Tile 0)
SetDACVOP
DATA UPLOADED
 Programming DAC 1_228 (DAC 1, Tile 0)...
SetMemType
SetDataPathMode 0 1 2 
SetMMCM 1 5 2 5 0 
SetNyquistZone
SetupFIFO
SetInterpolationFactor
SetMMCM 1 5 2 5 0 
IntrClr
SetupFIFO
MultiBand 1 0 1 4 1
SetMixerSettings
ResetNCOPhase
UpdateEvent
Set